# Pod 9: fit it

A **model** is a function with knobs. A **fit** turns the knobs until the model matches the data as closely as it can. Today you fit a straight line three ways: by eye, with `curve_fit`, and by turning the knobs yourself a small step at a time, which is **gradient descent**, the method behind every model you will hear of this decade.

Run the first cell; it must print a path containing `astr119`. Before every code cell: write your prediction on the sheet, run, compare.

**How to work the "your code here" cells:** one sentence of what you expect in a text cell, then type it or ask Claude for that cell only, read it, run, compare.

In [ ]:
import sys
print(sys.executable)   # must contain 'astr119'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## Part 1: three ways to fit a line, together

Five made-up measurements first, so the numbers are small enough to check by hand.

In [ ]:
# F1  Five points, and a line by eye. The residual is data minus model; the loss is the sum of the squared residuals.
xs = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
ys = np.array([2.1, 3.9, 6.2, 7.8, 10.1])

def line(x, slope, intercept):
    return slope * x + intercept

def loss(slope, intercept):
    residuals = ys - line(xs, slope, intercept)
    return np.sum(residuals**2)

for slope in (1.5, 2.0, 2.5):                      # guess the slope by eye; intercept 0
    print(slope, round(loss(slope, 0.0), 2))       # 14.06, 0.11, 13.66: the middle one is close

fig, ax = plt.subplots()
ax.scatter(xs, ys, label="data")
for slope in (1.5, 2.0, 2.5):
    ax.plot(xs, line(xs, slope, 0.0), label=f"slope {slope}")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
fig.savefig("fit_by_eye.png")

In [ ]:
# F2  Let the computer turn the knobs: curve_fit finds the slope and intercept with the smallest loss, and says how sure it is.
best, cov = curve_fit(line, xs, ys)
errors = np.sqrt(np.diag(cov))
print("slope, intercept:", best.round(3))          # [1.99 0.05]
print("uncertainties:   ", errors.round(3))
print("loss at the best fit:", round(loss(*best), 3))   # smaller than any guess above

residuals = ys - line(xs, *best)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.5))
a1.scatter(xs, ys); a1.plot(xs, line(xs, *best), "k-"); a1.set_xlabel("x"); a1.set_ylabel("y"); a1.set_title("the fit")
a2.scatter(xs, residuals); a2.axhline(0, color="k"); a2.set_xlabel("x"); a2.set_ylabel("data - model"); a2.set_title("the residuals")
fig.savefig("fit_residuals.png")

In [ ]:
# F3  Gradient descent: turn the knobs yourself, a small step downhill each time. The slope of the loss tells you which way is down.
slope, intercept = 0.0, 0.0                       # start anywhere
rate = 0.01                                       # how big a step (0.02 is already too big here: try it later and watch it explode)
history = []
for step in range(200):
    residuals = ys - line(xs, slope, intercept)
    d_slope = -2 * np.sum(residuals * xs)         # how the loss changes if the slope goes up a little
    d_intercept = -2 * np.sum(residuals)          # same for the intercept
    slope = slope - rate * d_slope                # step downhill
    intercept = intercept - rate * d_intercept
    history.append(loss(slope, intercept))
print("after 200 steps:", round(slope, 3), round(intercept, 3))   # 1.986 0.065: nearly curve_fit's answer; 1000 steps gives 1.990 0.050

fig, ax = plt.subplots()
ax.plot(history)
ax.set_yscale("log")
ax.set_xlabel("step")
ax.set_ylabel("loss")
fig.savefig("fit_loss.png")

## Task 1: Kepler's third law, from the catalog (the core)

For planets around Sun-like stars, Kepler's third law says $P^2 \propto a^3$: on log axes, $\log P = 1.5\,\log a + \text{const}$, and with $P$ in years and $a$ in AU the constant is 0. The catalog has 900 such planets. Fit the line and see whether a 17th-century law survives contact with 21st-century data.

In [ ]:
# 1.1  Read the catalog and keep the planets around Sun-like stars (star mass 0.9 to 1.1 suns) that have a period and a distance. Predict how many.
cat = pd.read_csv("../module-4/data/exoplanet_eu_catalog_2024-02.csv", low_memory=False)   # the catalog from Pod 8, one folder over
keep = cat["orbital_period"].notna() & cat["semi_major_axis"].notna() & cat["star_mass"].between(0.9, 1.1)
k = cat[keep]
print(len(k))
x = np.log10(k["semi_major_axis"].values)            # log of the distance, AU
y = np.log10(k["orbital_period"].values / 365.25)     # log of the period, years

In [ ]:
# 1.2  Look before you fit. Predict: a straight line? with what slope?
fig, ax = plt.subplots()
ax.scatter(x, y, s=6)
ax.set_xlabel("log10 semi-major axis (AU)")
ax.set_ylabel("log10 orbital period (years)")
fig.savefig("kepler_data.png")

In [ ]:
# 1.3  Fit the line with curve_fit. Predict the slope and the intercept before you run.
best, cov = curve_fit(line, x, y)
errors = np.sqrt(np.diag(cov))
print("slope, intercept:", best.round(4))
print("uncertainties:   ", errors.round(4))
residuals = y - line(x, *best)
print("residual std:", residuals.std().round(4))

In [ ]:
# 1.4  The residuals. Predict: scattered evenly, or a pattern? Then: who are the planets that sit far off the line?
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.scatter(x, y, s=6); a1.plot(x, line(x, *best), "k-", label=f"slope {best[0]:.3f}"); a1.legend()
a1.set_xlabel("log10 a (AU)"); a1.set_ylabel("log10 P (yr)")
a2.scatter(x, residuals, s=6); a2.axhline(0, color="k")
a2.set_xlabel("log10 a (AU)"); a2.set_ylabel("data - model")
fig.savefig("kepler_fit.png")

far = np.abs(residuals) > 0.2
print(k[far][["name", "orbital_period", "semi_major_axis", "star_mass"]])   # Your sentence: what is wrong with these rows?

## Task 2: turn the knobs yourself

Same data, no `curve_fit`: gradient descent, as in F3. Does it reach the same line? What does the learning rate do?

In [ ]:
# 2.1  Gradient descent on the Kepler data. Predict: does it reach the curve_fit answer? After how many steps?
slope, intercept = 0.0, 0.0
rate = 0.1                                        # try this first
history = []
for step in range(200):
    residuals = y - line(x, slope, intercept)
    d_slope = -2 * np.mean(residuals * x)         # mean instead of sum: the step size then does not depend on how many planets there are
    d_intercept = -2 * np.mean(residuals)
    slope = slope - rate * d_slope
    intercept = intercept - rate * d_intercept
    history.append(np.mean((y - line(x, slope, intercept))**2))
print("after 200 steps:", round(slope, 4), round(intercept, 4), "  (curve_fit gave", best.round(4), ")")

fig, ax = plt.subplots()
ax.plot(history)
ax.set_yscale("log")
ax.set_xlabel("step")
ax.set_ylabel("mean squared residual")
fig.savefig("kepler_loss.png")

In [ ]:
# 2.2  The learning rate. Predict what happens with rate = 0.01, then with rate = 1.0. Run each (change the number above, or copy the loop here).
# Your code here: the loss curves for three rates on one figure, labeled.


## Task 3: make it yours

Pick one as a pod, or combine, or invent something; each of you in your own notebook. Predict before each run.

- **The width, fitted.** Fit a bell curve to the histogram of `log10(planets["mass"])` as in Pod 7 (counts against bin centers, `gauss` with height, center, width, a starting guess). Is one bell enough? Show the residuals.
- **Mass and radius.** For planets with a period under 10 days and a mass between 0.3 and 3 Jupiter masses, fit radius against mass with a line. Does the slope mean anything? What does the scatter mean?
- **Not Sun-like.** Repeat Task 1 for stars of 0.3 to 0.5 solar masses. The slope should still be 1.5; the intercept moves. By how much, and why (Kepler's full law has the star's mass in it)?
- **A curve, not a line.** Fit `y = a * x**2 + b * x + c` to the five points of Part 1. Does the loss drop? Is the better fit a better model? (Three knobs for five points.)
- **Break it.** Give gradient descent a starting slope of 100, or a learning rate of 2. Describe what the loss does and why.
- **A combination, or something else** that fits a model to data and shows the residuals. Write what you did on the sheet.

In [ ]:
# Task 3: your code here


Before you upload: **Restart** and **Run All**, check it runs top to bottom. Then upload this notebook, with its outputs, and the photo of the sheet.